<a href="https://colab.research.google.com/github/IzzulAhmad/TUGAS-UTS-IZZUL-14022300077-6B-BIS-BIG-DATA/blob/main/UTS_Izzul_Ahmad_Fathoni_14022300077_6B_BIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.gojek.gopay',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers pandas

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(filename)

# Display the first few rows to verify
display(df.head())

In [ ]:
# Inisialisasi pipeline dengan model yang BENAR
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)

# Fungsi untuk mendapatkan sentimen
def get_sentiment(text):
    if pd.isna(text):
        return None, None
    result = sentiment_pipeline(text[:512])[0]  # RoBERTa punya batas 512 token
    return result['label'], result['score']

# Terapkan ke DataFrame
df[['sentiment_label', 'sentiment_score']] = df['content'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

# Tampilkan hasil
display(df.head())

Let's analyze the distribution of sentiment labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of sentiment labels
plt.figure(figsize=(8, 6))
sns.countplot(x='sentiment_label', data=df, palette='viridis')
plt.title('Distribution of Sentiment Labels')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()
